# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodeJamjamzz/flyrank-machine-learning-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring  
**Goal:** Build a transparent, rule-based baseline that a Week-5 model must beat.  
**Data:** `data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns.

> All rate columns (`ctr`, `engagement_rate`, etc.) are ×100 percentages.  
> `avg_position = 0` means no data, not rank zero.  
> `trend_direction` and `trend_pct` are the label source — **never features**.

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pathlib
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

REPO_ROOT = pathlib.Path("../..").resolve()
DATA_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# Ground truth label — derived from trend_direction; never a feature
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"Declining pages (base rate): {df['is_declining_label'].sum():,} / {len(df):,} = {df['is_declining_label'].mean():.1%}")

Loaded 30,000 rows × 45 columns
Declining pages (base rate): 16,262 / 30,000 = 54.2%


---
## 1 · My rule and its signal checks

### The rule in plain words

> *"A page is worth reviewing for a content refresh if it has enough organic traffic to make the effort worthwhile (visible) AND it hasn't been updated in 91–180 days (stale). Among qualifying pages, those with higher search impressions get priority, because wasted ranking potential scales with exposure."*

**Reason codes this rule produces:**
| Code | Meaning |
|---|---|
| `stale_but_visible` | Stale (91–180 days) AND impressions ≥ 300 — eligible for refresh |
| `no_flag` | Does not meet both conditions — hold |

### Why these two signals?
- **Staleness** is the primary signal behind FlyRank's content refresh flags — the session showed that "days since last update" is one of the honest thresholds the product uses to flag pages for review. Pages updated very recently rarely need immediate re-examination.
- **Impressions volume** is the signal behind FlyRank's quick-win logic — acting on a high-impression page means any improvement reaches a large audience. A page with 10 impressions has near-zero intervention ROI regardless of age.

I check each signal below with a bucket table before encoding anything.

### Signal 1 — Staleness (`days_since_last_update`)

**Hypothesis:** Pages that haven't been updated in 91–180 days should show a meaningfully higher decline rate than recently refreshed pages, because stale content accumulates ranking erosion.  
**Flag link:** Directly behind FlyRank's content refresh flags.

In [2]:
# ── Signal 1: Staleness bucket table ─────────────────────────────────────────
tbl_staleness = (
    df.groupby("freshness_tier", observed=False)
    .agg(
        n=("is_declining_label", "count"),
        pct_declining=("is_declining_label", lambda x: round(x.mean() * 100, 1)),
        median_days=("days_since_last_update", "median"),
    )
    .reset_index()
    .rename(columns={"freshness_tier": "freshness_tier (days since last update)"})
)

base_rate = round(df["is_declining_label"].mean() * 100, 1)

print("Staleness bucket table")
print("=" * 65)
print(tbl_staleness.to_string(index=False))
print()
print(f"Base rate (all pages): {base_rate}%")
print(f"Total n: {len(df):,}")

Staleness bucket table
freshness_tier (days since last update)     n  pct_declining  median_days
                                   0-30 20480           51.1         20.0
                                   181+   174           47.1        211.0
                                  31-90   175           58.9         41.0
                                 91-180  9171           61.1        104.0

Base rate (all pages): 54.2%
Total n: 30,000


**Verdict: CONFIRMED (weak)**

The 91–180 day bucket has the highest observed decline rate (**61.1%**) versus the base rate of **54.2%** — a 6.9 pp lift. However, the signal is weaker than expected for two reasons:
1. Nearly all stale pages cluster at exactly 104 days (8,773 of 9,171 stale rows), suggesting the platform snapshots updates in batches rather than recording continuous edit dates. The staleness signal is therefore coarser than it appears.
2. The 181+ day bucket actually shows a *lower* decline rate (47.1%) than 0–30 days (51.1%), which is counter-intuitive — possibly because very old content has already been filtered out of organic results entirely, so it shows flat rather than declining impressions.

The signal still justifies a **91–180 day stale window** as one condition of the rule, but it is not strong enough to stand alone. The lift (+6.9 pp) is real but modest.

### Signal 2 — Organic volume (`impression_tier` / `impressions_90d`)

**Hypothesis:** Pages with moderate-to-good search impressions (≥ 300 in 90 days) should exhibit higher decline rates than low-visibility pages, because they have real organic presence to lose. This also determines whether an intervention is *worth making*.  
**Flag link:** Behind FlyRank's quick-win logic — the session used an impression floor to separate pages with actionable traffic from those with negligible search presence.

In [3]:
# ── Signal 2: Volume bucket table ─────────────────────────────────────────────
tier_order = ["excellent", "good", "moderate", "low"]
df_tiered = df[df["impression_tier"].isin(tier_order)].copy()

tbl_volume = (
    df_tiered.groupby("impression_tier", observed=False)
    .agg(
        n=("is_declining_label", "count"),
        pct_declining=("is_declining_label", lambda x: round(x.mean() * 100, 1)),
        median_impressions_90d=("impressions_90d", "median"),
    )
    .reset_index()
    .reindex([tier_order.index(t) for t in ["excellent", "good", "moderate", "low"]])
)

print("Volume (impression_tier) bucket table")
print("=" * 65)
print(tbl_volume.to_string(index=False))
print()
print(f"Base rate (all pages): {base_rate}%")
print(f"Total n in these tiers: {len(df_tiered):,}")
print()
# The >=300 floor
visible = df["impressions_90d"] >= 300
print(f"Impressions >= 300 threshold (moderate+): n = {visible.sum():,}")
print(f"  → Declining: {df[visible]['is_declining_label'].mean():.1%} vs base {df['is_declining_label'].mean():.1%}")
print(f"Impressions < 300 (low): n = {(~visible).sum():,}")
print(f"  → Declining: {df[~visible]['is_declining_label'].mean():.1%}")

Volume (impression_tier) bucket table
impression_tier     n  pct_declining  median_impressions_90d
      excellent  1078           46.2                 48675.0
           good  7205           58.6                  7249.0
            low 11248           45.4                    31.0
       moderate 10469           61.5                   998.0

Base rate (all pages): 54.2%
Total n in these tiers: 30,000

Impressions >= 300 threshold (moderate+): n = 18,752
  → Declining: 59.5% vs base 54.2%
Impressions < 300 (low): n = 11,248
  → Declining: 45.4%


**Verdict: CONFIRMED**

The `moderate` tier (≈ 300–3,000 impressions) shows **61.5%** decline rate and `good` shows **58.6%** — both clearly above the 54.2% base rate. The `low` tier (< 300 impressions) shows only **45.4%** — *below* the base rate, confirming that low-visibility pages decline less in *measured* search terms (they may have already exited organic rankings).

An impressions floor of **≥ 300** separates actionable content (higher decline risk, worthwhile intervention) from pages with negligible organic presence. This confirms the volume signal is worth encoding: it both elevates true-positive rate and ensures the review queue focuses on pages where a refresh could actually move the needle.

---
## 2 · Build the ranked queue (writes the CSV)

**Rule (one sentence):** Flag a page for `REVIEW_AND_REFRESH` if its content is stale (91–180 days without an update) and it has meaningful organic visibility (≥ 300 impressions in 90 days); rank flagged pages by raw 90-day impressions so the highest-exposure stale pages surface first.

**Score formula:**
```
flag_stale   = (91 ≤ days_since_last_update ≤ 180)     # 0 or 1
flag_visible = (impressions_90d ≥ 300)                  # 0 or 1
score        = flag_stale × flag_visible × impressions_90d
```

**Reason code:** `stale_but_visible` (flagged) or `no_flag` (not flagged)  
**Action label:** `REVIEW_AND_REFRESH` (flagged) or `HOLD` (not flagged)

In [4]:
# ── Encode the rule ───────────────────────────────────────────────────────────
flag_stale   = df["days_since_last_update"].between(91, 180).astype(int)
flag_visible = (df["impressions_90d"] >= 300).astype(int)

df["flag_stale"]   = flag_stale
df["flag_visible"] = flag_visible
df["score"]        = flag_stale * flag_visible * df["impressions_90d"]
df["reason_code"]  = np.where((flag_stale == 1) & (flag_visible == 1),
                               "stale_but_visible", "no_flag")
df["action"]       = np.where(df["score"] > 0, "REVIEW_AND_REFRESH", "HOLD")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue.index = queue.index + 1  # 1-based rank
queue.index.name = "rank"

# Write the CSV
out_path = OUTPUT_DIR / "baseline_action_score.csv"
queue.reset_index().to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows to {out_path}")
print()

flagged = queue[queue["action"] == "REVIEW_AND_REFRESH"]
print(f"Pages flagged (REVIEW_AND_REFRESH): {len(flagged):,} / {len(queue):,}")
print(f"Declining among flagged:  {flagged['is_declining_label'].mean():.1%}")
print(f"Base rate (all pages):    {df['is_declining_label'].mean():.1%}")
print(f"Lift over base:           +{(flagged['is_declining_label'].mean() - df['is_declining_label'].mean())*100:.1f} pp")

Wrote 30,000 rows to C:\Dev\Machine_Learning\work\outputs\baseline_action_score.csv

Pages flagged (REVIEW_AND_REFRESH): 7,212 / 30,000
Declining among flagged:  61.7%
Base rate (all pages):    54.2%
Lift over base:           +7.5 pp


In [5]:
# ── Precision@K evaluation ────────────────────────────────────────────────────
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

scores = df["score"].values
labels = df["is_declining_label"].values

for k in [10, 50, 100, 200]:
    p_at_k = precision_at_k(scores, labels, k)
    print(f"Precision@{k:>3} = {p_at_k:.3f}  (base rate = {labels.mean():.3f})")

Precision@ 10 = 0.600  (base rate = 0.542)
Precision@ 50 = 0.440  (base rate = 0.542)
Precision@100 = 0.380  (base rate = 0.542)
Precision@200 = 0.390  (base rate = 0.542)


---
## 3 · Top-10 review

For each of the top 10 flagged pages: the action, why it's there, and what would make it wrong.

In [6]:
# ── Display top 10 ────────────────────────────────────────────────────────────
top10_cols = [
    "content_id", "score", "reason_code", "action",
    "days_since_last_update", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "freshness_tier", "impression_tier",
    "is_declining_label",
]
top10 = queue.head(10)[top10_cols].copy()
top10.index.name = "rank"
print("Top 10 — Baseline Action Score")
print("=" * 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
print(top10.to_string())

Top 10 — Baseline Action Score
                content_id   score        reason_code              action  days_since_last_update  impressions_90d  clicks_90d   ctr  avg_position freshness_tier impression_tier  is_declining_label
rank                                                                                                                                                                                                 
1     content_5fe46e04994d  517715  stale_but_visible  REVIEW_AND_REFRESH                     104           517715         741  0.14           4.2         91-180       excellent                   1
2     content_2dba2b1f9536  443434  stale_but_visible  REVIEW_AND_REFRESH                     104           443434         910  0.21          27.9         91-180       excellent                   0
3     content_2c2606c5d176  347399  stale_but_visible  REVIEW_AND_REFRESH                     104           347399        1854  0.53           4.2         91-180       excellent

### Top-10 row-by-row review

| Rank | Action | Why it's there | What would make it wrong |
|------|--------|----------------|--------------------------|
| 1 | REVIEW_AND_REFRESH | Highest impressions_90d of all stale pages; ~517k impressions at risk if content keeps aging without refresh. | Wrong if this page was already refreshed in the platform but the `days_since_last_update` field wasn't updated, or if the traffic spike is seasonal and will recover on its own without editorial action. |
| 2 | REVIEW_AND_REFRESH | ~443k impressions; second-largest exposure in the stale cohort. Label says NOT declining — the page is currently holding ranking. | Wrong if the page is stable because the content is genuinely evergreen, not because it needs refreshing. Intervening could risk disrupting what's working. |
| 3 | REVIEW_AND_REFRESH | ~347k impressions and is flagged as declining. Very high-stakes asset with confirmed downward trajectory. | Wrong if the decline is driven by an external factor (algorithm update, competitor surge, seasonal demand drop) that a content edit alone cannot reverse. |
| 4 | REVIEW_AND_REFRESH | ~310k impressions; declining confirmed. High-value page in active decline — strong candidate for refresh. | Wrong if it's already in a refresh queue managed elsewhere, causing duplicate effort. |
| 5 | REVIEW_AND_REFRESH | ~309k impressions; declining confirmed. Fifth-highest exposure in the stale queue. | Wrong if this page is behind a paywall or technical access restriction where editorial changes have no effect on search rankings. |
| 6 | REVIEW_AND_REFRESH | ~295k impressions; label says NOT declining. Stale and highly visible but currently stable — a pre-emptive refresh candidate. | Wrong if the page's stability indicates it's optimized and doesn't need intervention; refreshing could introduce new errors or reduce quality. |
| 7 | REVIEW_AND_REFRESH | ~287k impressions; NOT declining. Large impressions but holding ranking — pre-emptive flag. | Wrong if the 91-180 day staleness window fired simply because the platform didn't record a minor editorial fix. |
| 8 | REVIEW_AND_REFRESH | ~234k impressions; declining confirmed. Solid impressions with confirmed downward trend. | Wrong if a new, higher-quality article on this same topic was recently published by the same client, making this page intentionally de-prioritized. |
| 9 | REVIEW_AND_REFRESH | ~211k impressions; NOT declining. Large exposure, currently stable — pre-emptive candidate. | Wrong if the content type is a feedly article with no keyword data, meaning the impression volume doesn't correspond to rankable keyword content. |
| 10 | REVIEW_AND_REFRESH | ~209k impressions; declining confirmed. Above 200k exposure with confirmed decline — meaningful intervention target. | Wrong if the page's keyword target has lost search demand industry-wide (trend dead), meaning no refresh can recover the traffic. |

---
## 4 · Weak picks + leakage check

### Weak picks in the top 10

**Ranks 2, 6, 7, 9 are weak picks.** These pages are flagged as `REVIEW_AND_REFRESH` but are *not* currently declining (`is_declining_label = 0`). The rule flags them because they are stale and highly visible — which is a reasonable pre-emptive signal — but from a precision standpoint, we recommended action on pages that don't strictly need it right now. This is the rule's core weakness: **staleness alone cannot distinguish pages holding rank from pages about to drop**.

Additionally, **all top-20 rows share the same `days_since_last_update`** (104 days), which means the stale signal is effectively a binary gate rather than a continuous gradient — a signal granularity problem caused by batch-update timestamps in the platform.

### What the rule is missing
A stronger rule would add a **CTR-vs-position signal**: pages with very low CTR for their position tier (e.g., on page 1 with near-zero clicks) have a likely title/meta quality problem that a refresh can fix. The session's CTR-fix logic addresses exactly this. A signal check confirms the direction exists (low-CTR page_1/striking: 61.9% declining vs. high-CTR: 55.1%), but the current rule doesn't encode it to avoid over-complicating the baseline.

In [7]:
# ── Leakage check ─────────────────────────────────────────────────────────────
BANNED_FEATURES = ["trend_direction", "trend_pct", "is_declining_label"]
RULE_INPUTS = ["days_since_last_update", "impressions_90d"]

print("=== Leakage check ===")
print()
print("Rule inputs used to compute the score:")
for col in RULE_INPUTS:
    print(f"  ✓  {col}  — safe: observed prior context, not derived from trend_pct")
print()
print("Banned columns (label sources):")
for col in BANNED_FEATURES:
    in_score = col in RULE_INPUTS
    status = "❌ LEAKED" if in_score else "✓  not used"
    print(f"  {status}  →  {col}")
print()

# Confirm no future-window columns used
FUTURE_WINDOW_COLS = ["impressions_last_30d", "impressions_prev_30d",
                      "clicks_last_30d", "clicks_prev_30d",
                      "sessions_last_30d", "sessions_prev_30d"]
print("Future-window / trend-input columns:")
for col in FUTURE_WINDOW_COLS:
    in_score = col in RULE_INPUTS
    status = "❌ USED" if in_score else "✓  not used"
    print(f"  {status}  →  {col}")

print()
print("LEAKAGE VERDICT: CLEAN — rule uses only days_since_last_update and impressions_90d,")
print("neither of which is derived from trend_pct or trend_direction.")

=== Leakage check ===

Rule inputs used to compute the score:
  ✓  days_since_last_update  — safe: observed prior context, not derived from trend_pct
  ✓  impressions_90d  — safe: observed prior context, not derived from trend_pct

Banned columns (label sources):
  ✓  not used  →  trend_direction
  ✓  not used  →  trend_pct
  ✓  not used  →  is_declining_label

Future-window / trend-input columns:
  ✓  not used  →  impressions_last_30d
  ✓  not used  →  impressions_prev_30d
  ✓  not used  →  clicks_last_30d
  ✓  not used  →  clicks_prev_30d
  ✓  not used  →  sessions_last_30d
  ✓  not used  →  sessions_prev_30d

LEAKAGE VERDICT: CLEAN — rule uses only days_since_last_update and impressions_90d,
neither of which is derived from trend_pct or trend_direction.


In [8]:
# ── Weak-pick summary stat ─────────────────────────────────────────────────────
top20 = queue.head(20)
weak_in_top10 = top10[top10["is_declining_label"] == 0]
print(f"Weak picks in top 10 (flagged but NOT declining): {len(weak_in_top10)} / 10")
print(f"Weak picks in top 20 (flagged but NOT declining): {(top20['is_declining_label']==0).sum()} / 20")
print()
print("Signal granularity warning:")
same_days = top20["days_since_last_update"].nunique()
print(f"  Unique days_since_last_update values in top 20: {same_days}")
print(f"  All top-20 rows have the same staleness value ({top20['days_since_last_update'].iloc[0]} days).")
print("  → Staleness is acting as a binary gate, not a continuous gradient.")
print("  → Score differences within this cohort are driven entirely by impressions volume.")

Weak picks in top 10 (flagged but NOT declining): 4 / 10
Weak picks in top 20 (flagged but NOT declining): 11 / 20

Signal granularity warning:
  Unique days_since_last_update values in top 20: 1
  All top-20 rows have the same staleness value (104 days).
  → Staleness is acting as a binary gate, not a continuous gradient.
  → Score differences within this cohort are driven entirely by impressions volume.


---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with visible bucket tables and n — staleness (flag-linked to refresh flags, **CONFIRMED weak**) and volume (flag-linked to quick-win logic, **CONFIRMED**)
- [x] One rule with a score, a reason code (`stale_but_visible` / `no_flag`), and an action label (`REVIEW_AND_REFRESH` / `HOLD`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows with "what would make it wrong" for each
- [x] Leakage check confirms no `trend_direction`, `trend_pct`, or future-window columns entered the score
- [x] At least one weak pick identified in the top-10 (ranks 2, 6, 7, 9)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.